# TenSEAL Implementation ML - Two

In [1]:
import torch
import time
from torchvision import datasets
import torchvision.transforms as transforms
import numpy as np
import tenseal as ts

### Load Data

- Sets fixed random seed for reproducibility
- Defines batch size for data loading during training and testing
- Loads MNIST training/testing dataset (downloads if not present and converts images to tensors)
- Wraps the training/testing dataset in a DataLoader (loads data in mini-batches, shuffles the data at each epoch)

In [2]:
torch.manual_seed(73)

train_data = datasets.MNIST('data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST('data', train=False, download=True, transform=transforms.ToTensor())

batch_size = 64

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

### Non-Encrypted Model Pipeline

### Define CNN Model Compatible with FHE

- INIT: Convolution Layer 1 (conv1) -> Input 1 channel, output 4 channels
- INIT: Fully Connected Layer 1 (fc1) -> input 256, output 64
- INIT: Output Fully COnnected Layer (fc2) -> Input 64, output 10
- FORWARD: Applies convolution
- FORWARD: Applies square activation function (instead of ReLU for HE-compatibility)
- FORWARD: Flatten tensor to shape for fully connected layer
Square activation is used so it can be later implemented in encryption form

In [3]:
class ConvNet(torch.nn.Module):
    def __init__(self, hidden=64, output=10):
        super(ConvNet, self).__init__()        
        self.conv1 = torch.nn.Conv2d(1, 4, kernel_size=7, padding=0, stride=3)
        self.fc1 = torch.nn.Linear(256, hidden)
        self.fc2 = torch.nn.Linear(hidden, output)

    def forward(self, x):
        x = self.conv1(x)
        x = x * x
        x = x.view(-1, 256)
        x = self.fc1(x)
        x = x * x
        x = self.fc2(x)
        return x

### Training Non-Encrypted Model

- Initializes the model
- Defines the loss function for multi-class classification
- Defines optimizer (Adam)
- Sets the model to training mode
- Trains for 10 epochs
- Iterate through each batch in the training DataLoader
- Clear gradients from previous step
- Forward pass: compute predicted outputs
- Compute the loss between predictions and actual labels
- Backward pass: compute gradients
- Update model weights
- Accumulate batch loss
- Compute average loss over all batches in this epoch
- Sets model to evaluation mode (disables dropout, batchnorm)

In [4]:
def train(model, train_loader, criterion, optimizer, n_epochs=10):
    # model in training mode
    model.train()
    for epoch in range(1, n_epochs+1):

        train_loss = 0.0
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # calculate average losses
        train_loss = train_loss / len(train_loader)

        print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch, train_loss))
    
    # model in evaluation mode
    model.eval()
    return model

In [5]:
model = ConvNet()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model = train(model, train_loader, criterion, optimizer, 10)

Epoch: 1 	Training Loss: 0.397561
Epoch: 2 	Training Loss: 0.130699
Epoch: 3 	Training Loss: 0.088399
Epoch: 4 	Training Loss: 0.071318
Epoch: 5 	Training Loss: 0.058989
Epoch: 6 	Training Loss: 0.050542
Epoch: 7 	Training Loss: 0.044438
Epoch: 8 	Training Loss: 0.038258
Epoch: 9 	Training Loss: 0.034622
Epoch: 10 	Training Loss: 0.031742


### Testing Non-Encrypted Model

- Initialize test loss and accuracy tracking per class
- Set model to evaluation mode (disables dropout, batchnorm)
- Loop through test data in batches
- Forward pass: compute predictions
- Compute loss for this batch and acculate
- Get predicted class (index of max logit)
- Compare predictions to true labels
- Count correct predictions per class
- Calculates average test loss across all batches then prints accuracy
- Prints overall accuracy across all classes

In [6]:
test_loss = 0.0
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))

model.eval()

for data, target in test_loader:
    output = model(data)
    loss = criterion(output, target)
    test_loss += loss.item()
    _, pred = torch.max(output, 1)
    correct = np.squeeze(pred.eq(target.data.view_as(pred)))
    for i in range(len(target)):
        label = target.data[i]
        class_correct[label] += correct[i].item()
        class_total[label] += 1

test_loss = test_loss/len(test_loader)
print(f'Test Loss: {test_loss:.6f}\n')

for label in range(10):
    print(
        f'Test Accuracy of {label}: {int(100 * class_correct[label] / class_total[label])}% '
        f'({int(np.sum(class_correct[label]))}/{int(np.sum(class_total[label]))})'
    )

print(
    f'\nTest Accuracy (Overall): {int(100 * np.sum(class_correct) / np.sum(class_total))}% ' 
    f'({int(np.sum(class_correct))}/{int(np.sum(class_total))})'
)

Test Loss: 0.072263

Test Accuracy of 0: 98% (969/980)
Test Accuracy of 1: 99% (1129/1135)
Test Accuracy of 2: 98% (1013/1032)
Test Accuracy of 3: 99% (1000/1010)
Test Accuracy of 4: 98% (969/982)
Test Accuracy of 5: 98% (877/892)
Test Accuracy of 6: 98% (945/958)
Test Accuracy of 7: 97% (1006/1028)
Test Accuracy of 8: 97% (954/974)
Test Accuracy of 9: 95% (967/1009)

Test Accuracy (Overall): 98% (9829/10000)


## Encrypted Model Pipeline

### Define Encrypted CNN Model using FHE

- INIT: Extracts and reshapes the convolutional weights from trained model
- INIT: Converts from torch tensors to nested Python lists
- INIT: Extracts bias values for conv1
- INIT: Extracts and transposes weigths for the first and second fully connected layer (fc1, fc2)
- FORWARD: Homomorphic convolution layer using im2col transformation
- FORWARD: Performs convolution via matrix multiplication and add bias
- FORWARD: Packs all encrypted channels into a single CKKS vector for next layers
- FORWARD: Apply square activation (FHE-compatible non-linearity)
- FORWARD: FC1 layer performs matrix multiplication + bias
- FORWARD: FC2 layer outputs logic for each class

In [7]:
class EncConvNet:
    def __init__(self, torch_nn):
        self.conv1_weight = torch_nn.conv1.weight.data.view(
            torch_nn.conv1.out_channels, torch_nn.conv1.kernel_size[0],
            torch_nn.conv1.kernel_size[1]
        ).tolist()
        self.conv1_bias = torch_nn.conv1.bias.data.tolist()
        
        self.fc1_weight = torch_nn.fc1.weight.T.data.tolist()
        self.fc1_bias = torch_nn.fc1.bias.data.tolist()
        
        self.fc2_weight = torch_nn.fc2.weight.T.data.tolist()
        self.fc2_bias = torch_nn.fc2.bias.data.tolist()
        
        
    def forward(self, enc_x, windows_nb):
        enc_channels = []
        for kernel, bias in zip(self.conv1_weight, self.conv1_bias):
            y = enc_x.conv2d_im2col(kernel, windows_nb) + bias
            enc_channels.append(y)
        enc_x = ts.CKKSVector.pack_vectors(enc_channels)
        enc_x.square_()
        enc_x = enc_x.mm(self.fc1_weight) + self.fc1_bias
        enc_x.square_()
        enc_x = enc_x.mm(self.fc2_weight) + self.fc2_bias
        return enc_x
    
    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

### Testing Encrypted Model

- Initialize test loss and accuracy counters per class
- Track correct and total predictions per class
- Track total runtime of encrypted inference
- Loop through test set one sample at a time (batch_size=1)
- Encode and encrypt the input image
- Convert image to a list format and apply im2col encoding to simulate convolution
- Run forward pass on encrypted input
- Decrypt the result for evaluation
- Calculate loss and prediction accuracy
- Get predicted class (index of highest)
- Check if prediction matches true label
- Update class-specific accuracy counters
- Prints statistics (total time, accuracy per class, overall test accuracy)

In [8]:
def enc_test(context, enc_model, test_loader, criterion, kernel_shape, stride):
    print("[INFO] Starting encrypted inference test...")
    
    test_loss = 0.0
    class_correct = list(0. for i in range(10))
    class_total = list(0. for i in range(10))

    total_start = time.time()
    
    for idx, (data, target) in enumerate(test_loader):
        if idx == 1000 or idx == 2000 or idx == 3000 or idx == 4000 or \
        idx == 5000 or idx == 6000 or idx == 7000 or idx == 8000 or idx == 9000 or idx == 10000:
            print(f"\n[INFO] Processing sample {idx}/{len(test_loader)}")
        
        x_enc, windows_nb = ts.im2col_encoding(
            context, data.view(28, 28).tolist(), kernel_shape[0],
            kernel_shape[1], stride
        )

        enc_output = enc_model(x_enc, windows_nb) # FHE inference
        output = enc_output.decrypt() # decrypt encrypted output
        output = torch.tensor(output).view(1, -1)
        loss = criterion(output, target)
        test_loss += loss.item()
        _, pred = torch.max(output, 1)
        correct = np.squeeze(pred.eq(target.data.view_as(pred)))
        label = target.data[0]
        class_correct[label] += correct.item()
        class_total[label] += 1

    total_time = time.time() - total_start
    test_loss = test_loss / sum(class_total)

    print(f"\n[SUMMARY] Total Encrypted Inference Time: {total_time:.2f}s")
    print(f"[SUMMARY] Average Test Loss: {test_loss:.6f}\n")

    for label in range(10):
        if class_total[label] > 0:
            acc = 100 * class_correct[label] / class_total[label]
            print(
                f"Test Accuracy of {label}: {int(acc)}% "
                f"({int(class_correct[label])}/{int(class_total[label])})"
            )
        else:
            print(f"Test Accuracy of {label}: N/A (no samples)")

    overall_acc = 100 * np.sum(class_correct) / np.sum(class_total)
    print(
        f"\n[SUMMARY] Overall Test Accuracy: {int(overall_acc)}% "
        f"({int(np.sum(class_correct))}/{int(np.sum(class_total))})"
    )

### Prepare testing data

- Load testing dataset into DataLoader (batch_size=1, shuffle)
- Extract kernel shape from trained model's first convolutional. This is used during im2col encoding for encrypted convolution
- Extract the stride value used in the convolutional layer
- Define the scaling factor (in bits) for the CKKS encryption scheme. Higher scale = higher precision but also larger ciphertext size
- Create a TenSEAL context for the CKKS scheme -> poly_modulus_degree (defines ciphertext size and noise budget), coeff_mod_bit_sizes (defines modulus chain depth)
- Sets the global scale for all encrypted operations
- Generate Galois keys required for vector rotations

In [9]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=True)
kernel_shape = model.conv1.kernel_size
stride = model.conv1.stride[0]
bits_scale = 26

context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[31, bits_scale, bits_scale, bits_scale, bits_scale, bits_scale, bits_scale, 31]
)

context.global_scale = pow(2, bits_scale)
context.generate_galois_keys()

### Run Encrypted CNN Model

In [10]:
enc_model = EncConvNet(model)
enc_test(context, enc_model, test_loader, criterion, kernel_shape, stride)

[INFO] Starting encrypted inference test...

[INFO] Processing sample 1000/10000

[INFO] Processing sample 2000/10000

[INFO] Processing sample 3000/10000

[INFO] Processing sample 4000/10000

[INFO] Processing sample 5000/10000

[INFO] Processing sample 6000/10000

[INFO] Processing sample 7000/10000

[INFO] Processing sample 8000/10000

[INFO] Processing sample 9000/10000

[SUMMARY] Total Encrypted Inference Time: 4565.89s
[SUMMARY] Average Test Loss: 0.079023

Test Accuracy of 0: 98% (970/980)
Test Accuracy of 1: 99% (1130/1135)
Test Accuracy of 2: 98% (1013/1032)
Test Accuracy of 3: 99% (1001/1010)
Test Accuracy of 4: 98% (970/982)
Test Accuracy of 5: 98% (876/892)
Test Accuracy of 6: 98% (940/958)
Test Accuracy of 7: 97% (1006/1028)
Test Accuracy of 8: 97% (952/974)
Test Accuracy of 9: 95% (966/1009)

[SUMMARY] Overall Test Accuracy: 98% (9824/10000)
